# 40: Baseline training

Trains the baseline ResNet-18 on a release's shards, using the training-run config in `config/gold/training_runs/` named by `TRAINING_RUN_NAME`. The same `run_training` call works from a laptop (`python -m ml.train`, see `docs/running.md`).

Run after `31_export_gold_shards` for the same release. The first cell installs its packages (pinned to `pyproject.toml`; PyTorch makes this take a few minutes), so there's nothing to set up. Training refuses to start without a git commit to record. The run, its inputs and the model (with its signature) go to MLflow, and the model is registered in Unity Catalog if the config sets `registered_model_name` ([ADR 007](../docs/decisions/007-pin-data-and-preprocessing-per-training-run.md)).

In [ ]:
%pip install --quiet torch==2.13.0 torchvision==0.28.0 mosaicml-streaming==0.13.0 mlflow-skinny==3.15.2 scikit-learn==1.9.0


In [ ]:
%run ./_setup_env


In [ ]:
import mlflow

from data_platform.dataset_layout import load_yaml_config
from ml.train import resolve_shards_root, run_training
from ml.training_run import resolve_training_run

# Which training run to build -- one small YAML file per run under config/gold/training_runs/,
# mirroring config/gold/manifests/<name>.yaml's convention. To try a different
# dataset_version/preprocessing_version/hyperparameter combination, add a new YAML file there
# rather than editing values in place here.
# Set by the train job's parameters; a manual run uses these defaults.
dbutils.widgets.text("training_run_name", "sample-v1-resnet18")
dbutils.widgets.text("git_commit", "")
TRAINING_RUN_NAME = dbutils.widgets.get("training_run_name")

print(f"Training run: {TRAINING_RUN_NAME}")


## Load the training-run config

In [ ]:
TRAINING_RUN_SPEC = resolve_training_run(CONFIG_ROOT, TRAINING_RUN_NAME)

print(f"dataset_version: {TRAINING_RUN_SPEC.dataset_version}")
print(f"preprocessing_version: {TRAINING_RUN_SPEC.preprocessing_version}")
print(f"architecture: {TRAINING_RUN_SPEC.architecture}, pretrained: {TRAINING_RUN_SPEC.pretrained}")
print(f"batch_size: {TRAINING_RUN_SPEC.batch_size}, num_epochs: {TRAINING_RUN_SPEC.num_epochs}, learning_rate: {TRAINING_RUN_SPEC.learning_rate}")
print(f"label_values: {TRAINING_RUN_SPEC.label_values}")


## Find the shards

On Databricks this is the release's Volume path.

In [ ]:
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
SHARDS_ROOT = resolve_shards_root(None, STORAGE_CONFIG, TRAINING_RUN_SPEC.dataset_version)

print(f"Shards root: {SHARDS_ROOT}")


## Train

In [ ]:
RUN_ID = run_training(
    TRAINING_RUN_NAME,
    config_root=CONFIG_ROOT,
    shards_root=SHARDS_ROOT,
    git_commit=dbutils.widgets.get("git_commit") or None,
)

print(f"MLflow run_id: {RUN_ID}")


## Review

In [ ]:
run_metrics = mlflow.MlflowClient().get_run(RUN_ID).data.metrics
for metric_name, value in sorted(run_metrics.items()):
    print(f"{metric_name}: {value}")
